In [4]:
import pandas as pd
import numpy as np

# Path to file
file_path = "../data/raw/Food_insecurity_Raw_Survey_Responses.xlsx"

# Load workbook
excel_file = pd.ExcelFile(file_path)

# Print sheet names
print("Available sheets:")
for sheet in excel_file.sheet_names:
    print(f"- {sheet}")

Available sheets:
- user_location
- 2022-12-03_incentivised
- 2022-11-22_27_incentivised
- 2022-11-22_27_non_incentivised
- 2022_12_05_all_responses_cleane
- Coded 
- USDA_food_security_coding
- adult_household_fi
- Preliminary results


In [9]:
from pathlib import Path
# Read the required sheet
df = pd.read_excel(excel_file, sheet_name="2022_12_05_all_responses_cleane")

print("\nDataset shape:", df.shape)

display(df.head())
df.info()

missing_report = (
    df.isna()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
      .reset_index()
)
# Create directory if it doesn't exist
output_dir = Path("../data/missing")
output_dir.mkdir(parents=True, exist_ok=True)

# Save missing value report
missing_report.columns = ["column", "missing_percentage"]
print(missing_report)

missing_report.to_csv(output_dir / "missing_report.csv", index=False)

duplicate_count = df.duplicated().sum()
print("\nDuplicate rows:", duplicate_count)

print("\nResponse Time Summary:")
print(df["response_time"].describe())


Dataset shape: (2886, 100)


,#,"By agreeing to take part in this survey, you confirm that you have read, understood and agreed with the following statements. Please note that this survey will skip to the end if you disagree to take part: \n \n _I agree that data gathered in this study will be stored anonymously and securely, and will be used for research purposes only. _\n \n _I understand that my participation is voluntary and that I am free to withdraw at any time without giving reason. _\n \n _I understand that all personal information will remain confidentially within OLIO's database, and that no personally identifiable information will be shared with any third party, and that no data will be made available that can allow me to be personally identified in the results of this research. _\n \n _I am 18 years of age or older. _\n \n I agree to take part in this survey:",How would you describe your gender?,What range best describes your age group?,Do you have any physical or mental health conditions or illnesses that are limiting your day to day activities?,Which of the following best describes your household?,"In general, how satisfied are you with your life?","Compared to others in my neighbourhood, I think my life overall is...",How often do you feel that you lack companionship?,How often do you feel isolated from others or left out?,...,What is your annual household income before tax?,user_id,Start Date (UTC),Submit Date (UTC),response_time,survey_date,incentivised,lad_code,msoa_code,imd_decile
0,01g1puj0zbtyyl68d9l01g1pujrxajj5,1,Female,25 to 29,No,Living with partner/ spouse (NO children or ch...,4,Worse,Hardly ever or never,Hardly ever or never,...,"£37,901- £58,900 p.a.",513482,2022-11-22 15:36:00,2022-11-22 15:39:00,3.0,2022-11-22,1,E09000013,E02000387,7
1,01h0q6h87uvwo01h0qeji2vxnl4z78ik,1,Female,40 to 44,No,Living with partner/ spouse (NO children or ch...,6,Better,Hardly ever or never,Some of the time,...,"£37,901- £58,900 p.a.",4319274,2022-11-22 16:01:00,2022-11-22 16:07:00,6.0,2022-11-22,1,E09000015,E02006882,7
2,01nhwhotteqvsg5qfl301nhwho84uknz,1,Male,30 to 34,No,Living with other adult family members (i.e. a...,8,Better,Often,Some of the time,...,Prefer not to say,6198143,2022-11-22 18:50:00,2022-11-22 18:56:00,6.0,2022-11-22,1,E09000033,E02000981,7
3,020yzynec712xmojk020yzynd1hjo9pl,1,Female,55 to 59,No,Living with other adult family members (i.e. a...,9,Better,Hardly ever or never,Hardly ever or never,...,"£24,301- £37,900 p.a.",2358564,2022-11-25 09:15:00,2022-11-25 09:22:00,7.0,2022-11-22,1,E09000009,E02000276,5
4,029tb4078h9gvgwasyk029tb49nw0foj,1,Male,55 to 59,No,Living with other adults that are non-family m...,7,The same,Some of the time,Some of the time,...,"£24,301- £37,900 p.a.",2592808,2022-11-22 17:39:00,2022-11-22 17:42:00,3.0,2022-11-22,1,E09000029,E02000852,10


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2886 entries, 0 to 2885
Data columns (total 100 columns):
 #   Column                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    Non-Null Count  Dtype         
---  ------   

In [10]:
output_dir = Path("../data/unified_csv")
output_dir.mkdir(parents=True, exist_ok=True)

# Save the sheet as CSV
csv_path = output_dir / "Food_insecurity_Raw_Survey_Responses.csv"
df.to_csv(csv_path, index=False)

print(f"CSV saved successfully to:\n{csv_path.resolve()}")

CSV saved successfully to:
C:\Users\dell 5590 i7\Documents\GITHUB\vanguard-group\data\unified_csv\Food_insecurity_Raw_Survey_Responses.csv


In [12]:
import pandas as pd
from pathlib import Path

# Load CSV
csv_path = Path("../data/unified_csv/Food_insecurity_Raw_Survey_Responses.csv")
df = pd.read_csv(csv_path)

# Drop unwanted columns
drop_cols = ["user_id", "Start Date (UTC)", "Submit Date (UTC)", "#"]
df = df.drop(columns=[col for col in drop_cols if col in df.columns])

results = []

for col in df.columns:
    total = len(df)

    # missing is a class
    value_counts = df[col].value_counts(dropna=False)

    for value, count in value_counts.items():
        if pd.isna(value):
            class_name = "missing"
        else:
            class_name = value

        percentage = (count / total) * 100

        results.append({
            "column": col,
            "class": class_name,
            "count": count,
            "percentage": round(percentage, 2)
        })

# Create result DataFrame
classes_df = pd.DataFrame(results)

# Save output
output_path = Path("../data/unified_csv/column_classes.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

classes_df.to_csv(output_path, index=False)

print(f"Saved to: {output_path.resolve()}")
display(classes_df.head(20))

Saved to: C:\Users\dell 5590 i7\Documents\GITHUB\vanguard-group\data\unified_csv\column_classes.csv


,column,class,count,percentage
0,"By agreeing to take part in this survey, you c...",1,2886,100.00
1,How would you describe your gender?,Female,2163,74.95
2,How would you describe your gender?,Male,642,22.25
3,How would you describe your gender?,Prefer not to say,45,1.56
4,How would you describe your gender?,Non binary/ other,36,1.25
5,What range best describes your age group?,35 to 39,546,18.92
6,What range best describes your age group?,30 to 34,489,16.94
7,What range best describes your age group?,25 to 29,406,14.07
8,What range best describes your age group?,40 to 44,365,12.65
9,What range best describes your age group?,45 to 49,266,9.22
